# Joshi Part 7: Binomial Trees & Implied Volatility (Python)

Binomial tree pricing, American options, and implied volatility via Newton-Raphson.

In [ ]:
import math
import numpy as np

## 1. CRR Binomial Tree

In [ ]:
def binomial_european(payoff_fn, S, r, sigma, T, steps):
    dt = T / steps
    u = math.exp(sigma * math.sqrt(dt))
    d = 1.0 / u
    disc = math.exp(-r * dt)
    p = (math.exp(r * dt) - d) / (u - d)
    q = 1.0 - p
    
    # Terminal payoffs
    values = [payoff_fn(S * u**i * d**(steps - i)) for i in range(steps + 1)]
    
    # Backward induction
    for step in range(steps - 1, -1, -1):
        values = [disc * (p * values[i+1] + q * values[i]) for i in range(step + 1)]
    
    return values[0]

def binomial_american(payoff_fn, S, r, sigma, T, steps):
    dt = T / steps
    u = math.exp(sigma * math.sqrt(dt))
    d = 1.0 / u
    disc = math.exp(-r * dt)
    p = (math.exp(r * dt) - d) / (u - d)
    q = 1.0 - p
    
    values = [payoff_fn(S * u**i * d**(steps - i)) for i in range(steps + 1)]
    
    for step in range(steps - 1, -1, -1):
        for i in range(step + 1):
            continuation = disc * (p * values[i+1] + q * values[i])
            exercise = payoff_fn(S * u**i * d**(step - i))
            values[i] = max(continuation, exercise)
    
    return values[0]

call_payoff = lambda s: max(s - 100, 0)
put_payoff = lambda s: max(100 - s, 0)

for steps in [50, 100, 200, 500]:
    c = binomial_european(call_payoff, 100, 0.05, 0.2, 1.0, steps)
    p = binomial_european(put_payoff, 100, 0.05, 0.2, 1.0, steps)
    print(f"Steps={steps:>4}: Call={c:.4f}, Put={p:.4f}")

print(f"\nBS exact: Call=10.4506, Put=5.5735")

## 2. American Put Premium

In [ ]:
steps = 500
euro_put = binomial_european(put_payoff, 100, 0.05, 0.2, 1.0, steps)
amer_put = binomial_american(put_payoff, 100, 0.05, 0.2, 1.0, steps)

print(f"European Put: {euro_put:.4f}")
print(f"American Put: {amer_put:.4f}")
print(f"Early exercise premium: {amer_put - euro_put:.4f}")

## 3. Newton-Raphson Implied Volatility

In [ ]:
def normal_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def normal_pdf(x):
    return math.exp(-0.5 * x * x) / math.sqrt(2 * math.pi)

def bs_call(S, K, r, sigma, T):
    d1 = (math.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*math.sqrt(T))
    d2 = d1 - sigma*math.sqrt(T)
    return S*normal_cdf(d1) - K*math.exp(-r*T)*normal_cdf(d2)

def bs_vega(S, K, r, sigma, T):
    d1 = (math.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*math.sqrt(T))
    return S * normal_pdf(d1) * math.sqrt(T)

def implied_vol_newton(market_price, S, K, r, T, initial_vol=0.2, tol=1e-8, max_iter=100):
    vol = initial_vol
    for i in range(max_iter):
        price = bs_call(S, K, r, vol, T)
        v = bs_vega(S, K, r, vol, T)
        diff = price - market_price
        if abs(diff) < tol:
            return vol, i + 1
        if abs(v) < 1e-15:
            raise ValueError("Vega is zero")
        vol -= diff / v
    raise ValueError("Did not converge")

# Round-trip test: price at vol=25%, then recover
true_vol = 0.25
price = bs_call(100, 100, 0.05, true_vol, 1.0)
implied, iters = implied_vol_newton(price, 100, 100, 0.05, 1.0)
print(f"True vol:    {true_vol:.6f}")
print(f"Implied vol: {implied:.6f}")
print(f"Iterations:  {iters}")
print(f"Error:       {abs(implied - true_vol):.2e}")

## 4. Implied Volatility Smile

In [ ]:
# Generate a skewed vol smile
S, r, T = 100, 0.05, 1.0
strikes = np.linspace(80, 120, 9)

# Simulate market prices with a skew (lower strike → higher vol)
true_vols = 0.20 + 0.001 * (100 - strikes)  # simple skew
market_prices = [bs_call(S, K, r, v, T) for K, v in zip(strikes, true_vols)]

# Recover implied vols
print(f"{'Strike':>8} {'MktPrice':>10} {'TrueVol':>10} {'ImplVol':>10}")
print("-" * 42)
for K, mp, tv in zip(strikes, market_prices, true_vols):
    iv, _ = implied_vol_newton(mp, S, K, r, T)
    print(f"{K:>8.1f} {mp:>10.4f} {tv:>10.4f} {iv:>10.4f}")